# Test locale API PoliMillionaire

Notebook minimo per verificare che il client API estratto localmente funzioni: import, login, lista competizioni, leaderboard e smoke test opzionale su una partita.

Nota: la password e salvata in chiaro nel notebook per comodita di test locale. Non condividere questo file pubblicamente senza rimuoverla.

In [13]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        client_dir = candidate / "api_client" / "NLP_assignment_api_client" / "millionaire_client"
        if client_dir.exists():
            return candidate
    raise FileNotFoundError("Could not find api_client/NLP_assignment_api_client/millionaire_client")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CLIENT_PARENT = PROJECT_ROOT / "api_client" / "NLP_assignment_api_client"
SRC_DIR = PROJECT_ROOT / "project" / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if str(CLIENT_PARENT) not in sys.path:
    sys.path.insert(0, str(CLIENT_PARENT))

print("Project root:", PROJECT_ROOT)
print("Client path:", CLIENT_PARENT)
print("Source path:", SRC_DIR)

Project root: C:\Users\Tommaso\Code\NLP_polimi_26
Client path: C:\Users\Tommaso\Code\NLP_polimi_26\api_client\NLP_assignment_api_client


In [14]:
from millionaire_client import (
    MillionaireClient,
    AuthenticationError,
    MillionaireError,
    TimeoutError,
)

API_URL = "http://131.175.15.22:51111/"
USERNAME = "NeuroniNegroni"
PASSWORD = "TomGiuliaGio"

client = MillionaireClient(API_URL, timeout=30)

try:
    user = client.login(USERNAME, PASSWORD)
    print(f"Login OK: {user.username} (role: {user.role})")
except AuthenticationError as exc:
    print("Login failed:", exc)
    raise

Login OK: NeuroniNegroni (role: student)


## Competizioni disponibili

In [15]:
competitions = client.competitions.list_all()

for comp in competitions:
    print(f"{comp.id}: {comp.name} | max levels: {comp.max_levels} | questions: {comp.question_count}")

0: Entertainment | max levels: 15 | questions: 843
1: Ancient History and Politics | max levels: 15 | questions: 456
2: Science and Nature | max levels: 15 | questions: 5395
3: Maths | max levels: 15 | questions: 390


## Leaderboard di una competizione

Modifica `COMPETITION_ID` se vuoi controllare una competizione diversa.

In [26]:
COMPETITION_ID = competitions[3].id if competitions else 1

leaderboard = client.leaderboard.get(competition_id=COMPETITION_ID, limit=10)
print(f"Leaderboard: {leaderboard.competition.name}")

for i, entry in enumerate(leaderboard.entries, 1):
    marker = " <- you" if entry.username == USERNAME else ""
    print(f"{i}. {entry.username}: score={entry.score}, level={entry.reached_level}{marker}")

Leaderboard: Maths
1. socialistcatwillbite: score=1024000, level=15
2. Khashayar: score=1024000, level=15
3. hasi: score=64000, level=11
4. IreneM: score=8000, level=8
5. supreme_leader: score=8000, level=8
6. luca_bordin: score=1000, level=5
7. grepapetti: score=1000, level=5
8. Anonymous: score=1000, level=5
9. ialkbr: score=500, level=4
10. gioferola: score=300, level=3


## Smoke test opzionale: avvio partita

Questa cella avvia una partita vera e fa partire il timer della prima domanda. Lasciala disattivata finche non vuoi testare anche `client.game.start()`.

In [17]:
RUN_GAME_SMOKE_TEST = False

if RUN_GAME_SMOKE_TEST:
    game = client.game.start(competition_id=COMPETITION_ID)
    question = game.current_question

    print("Session ID:", game.session_id)
    print("Current level:", game.current_level)
    print("Time remaining:", game.time_remaining)
    print("Question:", question.text if question else None)

    if question:
        for opt in question.options:
            print(f"{opt.id}: {opt.text}")
else:
    print("Smoke test partita disattivato. Imposta RUN_GAME_SMOKE_TEST = True per avviare una partita.")

Smoke test partita disattivato. Imposta RUN_GAME_SMOKE_TEST = True per avviare una partita.


## Test first option con logging completo

Questa sezione gioca una partita scegliendo sempre la prima opzione. Stampa in tempo reale domanda, opzioni, risposta scelta, risultato e salva un log strutturato per analisi successive.

In [27]:
import csv
import json
import time
from datetime import datetime


def choose_first_option(question):
    return question.options[0].id


def print_question(question):
    print(question.text)
    for opt in question.options:
        print(f"{opt.id}: {opt.text}")


def play_first_option_with_full_logging(client, competition_id, max_questions=None):
    game = client.game.start(competition_id=competition_id)
    logs = []
    started_at = datetime.now().isoformat(timespec="seconds")

    print(f"Session ID: {game.session_id}")
    print(f"Competition: {game.state.competition.name}")
    print(f"Started at: {started_at}")

    question_counter = 0

    while game.in_progress:
        if max_questions is not None and question_counter >= max_questions:
            print(f"\nStopped after max_questions={max_questions}. Game session is still open on the server.")
            break

        question = game.current_question
        if question is None:
            print("No active question. Game may have ended.")
            break

        question_counter += 1
        level_before = game.current_level
        earned_before = game.earned_amount
        time_remaining_before = game.time_remaining
        options = [{"id": opt.id, "text": opt.text} for opt in question.options]

        print("\n" + "=" * 80)
        print(f"Question #{question_counter} | Level {level_before}")
        print(f"Earned before: {earned_before}")
        print(f"Time remaining before answer: {time_remaining_before:.2f}s" if time_remaining_before is not None else "Time remaining before answer: n/a")
        print("\nQUESTION")
        print(question.text)
        print("\nOPTIONS")
        for opt in question.options:
            print(f"{opt.id}. {opt.text}")

        decision_start = time.perf_counter()
        chosen_option_id = choose_first_option(question)
        decision_latency = time.perf_counter() - decision_start
        chosen_option = question.get_option_by_id(chosen_option_id)

        print("\nFIRST OPTION ANSWER")
        print(f"Chosen id: {chosen_option_id}")
        print(f"Chosen text: {chosen_option.text if chosen_option else None}")
        print(f"Decision latency: {decision_latency:.4f}s")

        submit_start = time.perf_counter()
        error_message = None
        try:
            result = game.answer(chosen_option_id)
            submit_latency = time.perf_counter() - submit_start
        except Exception as exc:
            submit_latency = time.perf_counter() - submit_start
            result = None
            error_message = repr(exc)

        if result is not None:
            print("\nRESULT")
            print(f"Correct: {result.correct}")
            print(f"Timed out: {result.timed_out}")
            print(f"Game over: {result.game_over}")
            print(f"Earned after: {result.earned_amount}")
            print(f"Reached/current level: {result.reached_level or result.current_level}")
        else:
            print("\nERROR")
            print(error_message)

        log_row = {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "strategy": "first_option",
            "session_id": game.session_id,
            "competition_id": competition_id,
            "competition_name": game.state.competition.name,
            "question_number": question_counter,
            "question_id": question.id,
            "level": level_before,
            "question_text": question.text,
            "options_json": json.dumps(options, ensure_ascii=False),
            "chosen_option_id": chosen_option_id,
            "chosen_option_text": chosen_option.text if chosen_option else None,
            "time_remaining_before": time_remaining_before,
            "decision_latency_seconds": decision_latency,
            "submit_latency_seconds": submit_latency,
            "total_latency_seconds": decision_latency + submit_latency,
            "earned_before": earned_before,
            "correct": result.correct if result else None,
            "timed_out": result.timed_out if result else None,
            "game_over": result.game_over if result else None,
            "earned_after": result.earned_amount if result else None,
            "result_status": result.status if result else None,
            "current_level_after": result.current_level if result else None,
            "reached_level": result.reached_level if result else None,
            "error_message": error_message,
        }
        logs.append(log_row)

        if error_message is not None:
            break

    print("\n" + "=" * 80)
    print(f"Logged rows: {len(logs)}")
    return logs


def save_logs_csv(logs, path):
    if not logs:
        print("No logs to save.")
        return

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    file_exists = path.exists() and path.stat().st_size > 0

    with path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(logs[0].keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerows(logs)

    action = "Appended" if file_exists else "Created"
    print(f"{action} {len(logs)} rows in: {path}")

## Esegui first option test

Imposta `RUN_FIRST_OPTION_TEST = True` per giocare davvero. Ogni risposta viene inviata al server.

In [33]:
RUN_FIRST_OPTION_TEST = True
MAX_QUESTIONS = None  # usa un numero, per esempio 3, per fermarti prima della fine

if RUN_FIRST_OPTION_TEST:
    first_option_logs = play_first_option_with_full_logging(
        client=client,
        competition_id=COMPETITION_ID,
        max_questions=MAX_QUESTIONS,
    )

    output_path = PROJECT_ROOT / "logs" / f"first_option_comp_{COMPETITION_ID}.csv"
    save_logs_csv(first_option_logs, output_path)
else:
    print("First option test disattivato. Imposta RUN_FIRST_OPTION_TEST = True per eseguirlo.")

Session ID: 4555
Competition: Maths
Started at: 2026-04-27T16:26:39

Question #1 | Level 1
Earned before: 0
Time remaining before answer: 30.00s

QUESTION
Suppose there is a correlation of r = 0.9 between number of hours per day students study and GPAs. Which of the following is a reasonable conclusion?

OPTIONS
0. 90% of students who receive high grades study a lot.
1. 90% of students who study receive high grades.
2. 81% of the variation in GPAs can be explained by variation in number of study hours per day.
3. 90% of the variation in GPAs can be explained by variation in number of study hours per day.

FIRST OPTION ANSWER
Chosen id: 0
Chosen text: 90% of students who receive high grades study a lot.
Decision latency: 0.0000s

RESULT
Correct: False
Timed out: False
Game over: True
Earned after: 0
Reached/current level: None

Logged rows: 1
Appended 1 rows in: C:\Users\Tommaso\Code\NLP_polimi_26\logs\first_option_comp_3.csv
